# 📂 Convert EML Files to CSV

This notebook walks through the first stage of the email archiving pipeline: **recursively scanning a folder of `.eml` files and converting them into a structured flat CSV file** that downstream notebooks can work with.

The input is a directory of converted `.eml` files (exported from legacy `.pst` archive files). The output is a single CSV with one row per email, capturing the filename, full file path, and extracted plain-text body content.

> **Note:** All file paths in this notebook use generic placeholders. Update the `root_folder` and `output_csv` variables to match your local directory structure before running.


## 📦 Step 1: Import Libraries

We use three standard libraries:
- **`os`** — for walking the directory tree and building file paths
- **`email`** — Python's built-in library for parsing `.eml` files (RFC 2822 format)
- **`pandas`** — for collecting results into a DataFrame and writing the output CSV


In [ ]:
import os
import email
import pandas as pd

## 🔧 Step 2: Define the EML Text Extraction Function

The `extract_email_text()` function opens a single `.eml` file and extracts its plain-text body content. It handles two cases:

**Multipart emails** (e.g., emails with both a plain-text and HTML version, or with attachments):
- The function walks all MIME parts using `msg.walk()`
- It collects only `text/plain` parts that are not attachments (i.e., no `Content-Disposition` header)
- Multiple text parts are concatenated with a newline separator

**Single-part emails** (plain text only):
- The payload is decoded directly using the charset declared in the email headers, falling back to UTF-8 if none is specified

**Error handling:**
- File read errors are caught and printed without stopping the pipeline
- Decoding errors for individual MIME parts are caught and printed without discarding the rest of the email
- `errors='replace'` is used throughout so that malformed or unexpected byte sequences produce a replacement character rather than raising an exception — important for emails spanning 20+ years with inconsistent encoding practices


In [ ]:
def extract_email_text(file_path):
    """
    Extract the plain-text body from a single .eml file.

    Handles both single-part and multipart MIME emails. For multipart messages,
    only text/plain parts without a Content-Disposition header (i.e., not
    attachments) are included. Character encoding is read from the email headers
    and falls back to UTF-8 if not specified.

    Parameters
    ----------
    file_path : str
        Absolute path to the .eml file to parse.

    Returns
    -------
    str
        The extracted plain-text content, stripped of leading/trailing whitespace.
        Returns an empty string if the file cannot be read or contains no text.
    """
    text_content = ""

    # --- Open and parse the .eml file ---
    try:
        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            msg = email.message_from_file(f)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return text_content

    # --- Extract text from multipart or single-part message ---
    if msg.is_multipart():
        for part in msg.walk():
            # Only collect text/plain parts that are not file attachments
            if part.get_content_type() == "text/plain" and not part.get('Content-Disposition'):
                try:
                    charset = part.get_content_charset() or 'utf-8'
                    part_text = part.get_payload(decode=True).decode(charset, errors='replace')
                    text_content += part_text + "\n"
                except Exception as e:
                    print(f"Error decoding part of {file_path}: {e}")
    else:
        # Single-part email — decode the payload directly
        try:
            payload = msg.get_payload(decode=True)
            if payload:
                charset = msg.get_content_charset() or 'utf-8'
                text_content = payload.decode(charset, errors='replace')
            else:
                text_content = ""
        except Exception as e:
            print(f"Error decoding {file_path}: {e}")

    return text_content.strip()

## 📁 Step 3: Configure Input and Output Paths

Set `root_folder` to the top-level directory containing your converted `.eml` files. The script will scan all subdirectories recursively, so the folder can have any depth of nesting — the full path to each file will be preserved in the output.

Set `output_csv` to the desired location for the resulting CSV file. Make sure the target directory exists and that you have write permissions to it.

> **Tip for Windows users:** Use a raw string (`r"..."`) for paths, or double up backslashes (`"C:\\path\\to\\folder"`), to avoid Python treating backslashes as escape characters.


In [ ]:
# ── Configure these paths before running ──────────────────────────────────────

# Root directory containing the converted .eml archive files
# Update this to point to your local archive folder
root_folder = r"C:\Emails_Archiving\[ARCHIVE_FOLDER_NAME]"

# Output path for the resulting CSV file
# Make sure this directory exists and is writable
output_csv = r"C:\Emails_Archiving\emails_parsed.csv"

print(f"Root folder : {root_folder}")
print(f"Output CSV  : {output_csv}")

## 🔍 Step 4: Walk the Archive and Extract Email Content

This cell recursively walks the `root_folder` directory tree using `os.walk()`. For every file with a `.eml` extension (case-insensitive), it:

1. Builds the full absolute file path
2. Calls `extract_email_text()` to parse the plain-text body
3. Appends a dictionary record with three fields to the `emails_data` list:
   - `file_name` — the filename only (e.g., `20030610-1444 Subject line.eml`)
   - `file_path` — the full absolute path (used downstream to reconstruct folder hierarchy)
   - `email_text` — the extracted plain-text body content

After the walk is complete, the list is converted to a pandas DataFrame.

**Why store the full path?** The folder hierarchy in the original archive carries meaningful metadata — folder names like `Finance- Budget`, `Admin- Safety`, or `APP- Sys- Water Treatment Genl` describe the original organizational context of each email. Downstream notebooks split this path into individual `Folder_N` columns to make the hierarchy queryable.


In [ ]:
# Collect email data from all .eml files under root_folder
emails_data = []

for dirpath, dirnames, filenames in os.walk(root_folder):
    for filename in filenames:
        if filename.lower().endswith(".eml"):
            file_path = os.path.join(dirpath, filename)
            email_text = extract_email_text(file_path)
            emails_data.append({
                "file_name": filename,
                "file_path": file_path,
                "email_text": email_text
            })

# Build the DataFrame
df_emails = pd.DataFrame(emails_data)

print(f"Total .eml files found and parsed: {len(df_emails):,}")
print(f"\nDataFrame shape: {df_emails.shape}")
print(f"\nColumn names: {list(df_emails.columns)}")
print(f"\nSample file names (first 5):")
print(df_emails["file_name"].head().to_string())

**Example output:**

```
Total .eml files found and parsed: 44,677

DataFrame shape: (44677, 3)

Column names: ['file_name', 'file_path', 'email_text']

Sample file names (first 5):
0    20030610-1444 Subject line A.eml
1    20030612-1350 RE- Subject line B.eml
2    20030616-0821 Subject line C.eml
3    20030618-0848 RE- Subject line D.eml
4    20030625-1320 Subject line E.eml
```

> The archive produced **44,677 individual `.eml` files** across the full PST folder hierarchy.


## 💾 Step 5: Save to CSV

Write the DataFrame to a CSV file at `output_csv`. We use `index=False` to omit the pandas row index from the output — the downstream pipeline does not rely on it, and it keeps the file cleaner.

> **Common issue:** If you get a `PermissionError` here, it usually means either:
> - The target directory does not exist yet (create it first with `os.makedirs(...)`)
> - The CSV file is already open in another program (e.g., Excel)
> - You do not have write permissions to the target path


In [ ]:
# Create output directory if it doesn't already exist
os.makedirs(os.path.dirname(output_csv), exist_ok=True)

# Save to CSV
df_emails.to_csv(output_csv, index=False)

print(f"✅ Parsed {len(df_emails):,} emails and saved to:")
print(f"   {output_csv}")

✅ Parsed 44,677 emails and saved to:
   C:\Emails_Archiving\emails_parsed.csv


## 🔎 Step 6: Spot-Check the Output

Before passing this file to the next notebook, do a quick sanity check on the output. The cells below verify that the CSV was written correctly and that the content looks as expected.


In [ ]:
# Reload from disk to confirm the file was written correctly
df_check = pd.read_csv(output_csv)

print(f"Rows : {len(df_check):,}")
print(f"Cols : {len(df_check.columns)}")
print(f"\nNull counts per column:")
print(df_check.isnull().sum())
print(f"\nEmpty email_text count: {(df_check['email_text'] == '').sum()}")

In [ ]:
# Preview the first few rows
df_check.head()

In [ ]:
# Check the distribution of email text lengths to spot any parsing anomalies
df_check["text_length"] = df_check["email_text"].fillna("").str.len()

print("Email body length statistics (characters):")
print(df_check["text_length"].describe().round(1).to_string())
print(f"\nEmails with no body content (length = 0): {(df_check['text_length'] == 0).sum():,}")
print(f"Emails with very short body (<20 chars)  : {(df_check['text_length'] < 20).sum():,}")

---

## 📝 Notes on the Output Format

### Output columns

| Column | Description |
|---|---|
| `file_name` | The `.eml` filename — typically encodes date and subject (e.g., `20030610-1444 Subject line.eml`) |
| `file_path` | Full absolute path to the source file, including the PST folder hierarchy |
| `email_text` | Extracted plain-text body content. May be empty if the email had no text/plain part (e.g., HTML-only or attachment-only messages) |

### Filename convention

Email filenames in this archive follow the pattern `YYYYMMDD-HHMM Subject line.eml`. This encodes both the date/time and subject of the original message, making the filenames chronologically sortable without parsing headers.

### What this does NOT capture

This notebook extracts only the **plain-text body** of each email. It does not capture:
- Email headers (From, To, Cc, Subject, Date, Importance) — these are parsed in the next notebook
- HTML email content — only `text/plain` MIME parts are extracted
- Attachments — these are skipped by filtering on `Content-Disposition`
- Metadata embedded in the PST folder structure beyond what is encoded in the file path

All of these are handled in the downstream notebook `Email_Archiving_Data_Preparation_v1_CLEAN.ipynb`.

---

## ▶️ Next Step

Pass the output CSV to **`Email_Archiving_Data_Preparation_v1_CLEAN.ipynb`** for:
- Folder hierarchy extraction from file paths
- Email chain splitting
- Header parsing (From, To, Cc, Subject, etc.)
- PII detection and anonymization
- LLM-based summarization and topic modeling
